# Unsloth Core: Full NPC Pipeline for Colab

Run the complete **Unsloth Core** NPC fine-tuning pipeline -- from dataset generation through
sanitization, quality evaluation, training, and GGUF export.

### Pipeline Stages

| Stage | Description |
|-------|-------------|
| **1. Dataset Generation** | Create training data via template, Ollama, or API (OpenAI / Anthropic) |
| **2. Sanitization** | Clean ChatML format, remove AI artifacts, validate structure |
| **3. Quality Gate** | Optional DeepEval evaluation using a local Ollama judge model |
| **4. Training + Export** | Unsloth LoRA fine-tuning with inline GGUF export |
| **5. Download** | Download GGUF models and quality reports to your machine |

> **Requirements:** Google Colab with a GPU runtime (T4, L4, A100, or similar).
> If you are not running in Colab, open this notebook at
> [colab.research.google.com](https://colab.research.google.com/).


In [ ]:
# @title ⚙️ Pipeline Configuration
# @markdown Configure the NPC pipeline parameters before running the stages below.

# --- NPC Identity ---
NPC_KEY = "history_guide"  # @param {type:"string"}
# @markdown The NPC key in snake_case (e.g. `history_guide`, `chef_assistant`).

# --- Dataset Technique ---
TECHNIQUE = "template"  # @param ["template", "ollama", "openai", "anthropic"]
# @markdown - **template**: Fast deterministic generation (no external deps)
# @markdown - **ollama**: Local LLM generation via Ollama (needs GPU memory)
# @markdown - **openai**: OpenAI API generation (requires `OPENAI_API_KEY` secret)
# @markdown - **anthropic**: Anthropic API generation (requires `ANTHROPIC_API_KEY` secret)

# --- Training Preset ---
TRAINING_PRESET = "fast-3b"  # @param {type:"string"}
# @markdown Presets: `fast-3b` (standard), `safe-any` (low VRAM),
# @markdown `smoke` (debug), `quality-1.7b`, `fast-1.7b`, `fast-0.5b`

# --- Ollama Judge Model ---
OLLAMA_JUDGE_MODEL = "qwen2.5:7b"  # @param {type:"string"}
# @markdown Used for: (a) quality gate evaluation, (b) generation when technique=ollama.

# --- Skip Flags ---
# @markdown Check to skip dataset generation (use existing data):
SKIP_GENERATION = False  # @param {type:"boolean"}

print("Configuration loaded:")
print(f"  NPC_KEY           = {NPC_KEY}")
print(f"  TECHNIQUE         = {TECHNIQUE}")
print(f"  TRAINING_PRESET   = {TRAINING_PRESET}")
print(f"  OLLAMA_JUDGE_MODEL = {OLLAMA_JUDGE_MODEL}")
print(f"  SKIP_GENERATION   = {SKIP_GENERATION}")


In [ ]:
# @title 📦 Setup: Clone Repository & Install Dependencies
# @markdown Mounts Google Drive, clones the repo, installs Unsloth and other deps.

import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/athargamedev/Unsloth_Core.git"
DRIVE_REPO_DIR = "/content/drive/MyDrive/Unsloth_Core"
FALLBACK_REPO_DIR = "/content/Unsloth_Core"

# --- Detect Runtime ---
is_colab = False
try:
    import google.colab  # type: ignore
    is_colab = True
except ImportError:
    pass

if is_colab:
    print("Running in Google Colab.")
    repo_dir = DRIVE_REPO_DIR
    use_persistent = True
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
        if os.path.exists("/content/drive/MyDrive"):
            print("Google Drive mounted. Using persistent storage:", repo_dir)
        else:
            raise OSError("Google Drive mounted but MyDrive is not accessible.")
    except Exception as e:
        repo_dir = FALLBACK_REPO_DIR
        use_persistent = False
        print(f"Drive mount failed: {e}")
        print("Using ephemeral storage:", repo_dir)

    # Clone or pull the repository
    try:
        if not os.path.exists(repo_dir):
            os.makedirs(os.path.dirname(repo_dir), exist_ok=True)
            subprocess.run(["git", "clone", REPO_URL, repo_dir], check=True)
            print("Cloned repository.")
        else:
            git_dir = os.path.join(repo_dir, ".git")
            if os.path.exists(git_dir) and os.path.isdir(git_dir):
                orig = os.getcwd()
                os.chdir(repo_dir)
                subprocess.run(["git", "pull"], check=False)
                os.chdir(orig)
                print("Pulled latest changes.")
            else:
                import shutil
                shutil.rmtree(repo_dir)
                subprocess.run(["git", "clone", REPO_URL, repo_dir], check=True)
                print("Re-cloned repository (existing dir was not a git repo).")
    except OSError as e:
        print(f"Filesystem error during clone: {e}")
        if use_persistent:
            print("Falling back to ephemeral storage...")
            repo_dir = FALLBACK_REPO_DIR
            if not os.path.exists(repo_dir):
                subprocess.run(["git", "clone", REPO_URL, repo_dir], check=True)

    os.chdir(repo_dir)
    print("Changed to repo root:", os.getcwd())

    # Install Unsloth and dependencies
    print("\nInstalling Unsloth and dependencies...")
    subprocess.run(
        ["pip", "install", "--no-deps", "-q",
         "trl<0.9.0", "peft", "accelerate", "bitsandbytes", "xformers"],
        check=True,
    )
    subprocess.run(
        ["pip", "install", "-q",
         "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"],
        check=True,
    )
    subprocess.run(
        ["pip", "install", "-q",
         "gguf", "sentencepiece", "deepeval"],
        check=True,
    )
    print("All dependencies installed.")

else:
    print("=" * 60)
    print("NOT RUNNING IN GOOGLE COLAB.")
    print("Open this notebook at https://colab.research.google.com/")
    print("with a GPU runtime for the full pipeline.")
    print("=" * 60)
    print("Continuing in local environment (limited support).")
    # Attempt to find local repo root
    curr = Path(os.getcwd()).resolve()
    repo_dir = None
    for parent in [curr] + list(curr.parents):
        if (parent / "ucore").exists() and (parent / "scripts").exists():
            repo_dir = parent
            break
    if repo_dir:
        print("Found local repo root:", repo_dir)
        os.chdir(str(repo_dir))
    else:
        print("Warning: Could not find local repo root.")

print("\nWorking directory:", os.getcwd())


In [ ]:
# @title Stage 1: Dataset Generation
# @markdown Generate a training dataset using the configured technique.
# @markdown Edit `SKIP_GENERATION` in the config cell to skip this stage.

import os
import sys
import subprocess
import time
from pathlib import Path

python_bin = sys.executable
spec_path = f"subjects/{NPC_KEY}.json"
dataset_dir = f"subjects/datasets/{NPC_KEY}/{TECHNIQUE}"
train_jsonl = f"{dataset_dir}/train.jsonl"

print("=" * 60)
print("STAGE 1: Dataset Generation")
print(f"  NPC:       {NPC_KEY}")
print(f"  Technique: {TECHNIQUE}")
print(f"  Spec:      {spec_path}")
print(f"  Output:    {train_jsonl}")
print("=" * 60)

# Respect skip flag
if SKIP_GENERATION:
    print("SKIP_GENERATION is True. Skipping dataset creation.")
    if os.path.exists(train_jsonl):
        print(f"Existing dataset found at: {train_jsonl}")
    else:
        print(f"No dataset found at {train_jsonl} -- you may need to generate one.")
    print("Stage 1 skipped.")
    raise SystemExit(0)

# --- Template ---
if TECHNIQUE == "template":
    cmd = [python_bin, "scripts/generate_dataset.py",
           spec_path, "--technique", "template"]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

# --- Ollama ---
elif TECHNIQUE == "ollama":
    # Install Ollama if not present
    try:
        subprocess.run(["ollama", "--version"], check=True,
                       capture_output=True)
        print("Ollama is already installed.")
    except (FileNotFoundError, subprocess.CalledProcessError):
        print("Installing Ollama...")
        subprocess.run(
            "curl -fsSL https://ollama.com/install.sh | sh",
            shell=True, check=True, timeout=120,
        )
        print("Ollama installed.")

    # Ensure Ollama server is running
    try:
        subprocess.run(["ollama", "list"], check=True,
                       capture_output=True, timeout=10)
    except Exception:
        print("Starting Ollama server in background...")
        subprocess.Popen(
            ["ollama", "serve"],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        time.sleep(5)

    # Pull model
    print(f"Pulling model: {OLLAMA_JUDGE_MODEL}...")
    subprocess.run(["ollama", "pull", OLLAMA_JUDGE_MODEL], check=True,
                   timeout=300)

    # Generate
    cmd = [
        python_bin, "scripts/generate_dataset_ollama.py",
        spec_path,
        "--model", OLLAMA_JUDGE_MODEL,
        "--batch-size", "2",
        "--check-health",
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

# --- OpenAI / Anthropic ---
elif TECHNIQUE in ("openai", "anthropic"):
    api_key_var = "OPENAI_API_KEY" if TECHNIQUE == "openai" else "ANTHROPIC_API_KEY"
    api_key = os.environ.get(api_key_var)

    # Try Colab secrets
    if not api_key:
        try:
            from google.colab import userdata  # type: ignore
            api_key = userdata.get(api_key_var)
        except Exception:
            pass

    if not api_key:
        print(f"ERROR: {api_key_var} not found.")
        print("Set it as a Colab secret (key icon in the left sidebar)")
        print("or as an environment variable, then re-run this cell.")
        print("Skipping generation.")
        raise SystemExit(1)

    os.environ[api_key_var] = api_key
    model = ("gpt-4o-mini" if TECHNIQUE == "openai"
             else "claude-3-5-haiku-latest")
    cmd = [
        python_bin, "scripts/generate_dataset.py",
        spec_path,
        "--technique", TECHNIQUE,
        "--model", model,
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

else:
    print(f"Unknown technique: {TECHNIQUE}")
    print("Supported: template, ollama, openai, anthropic")
    raise SystemExit(1)

# Verify
if os.path.exists(train_jsonl):
    size_kb = os.path.getsize(train_jsonl) / 1024
    print(f"\nSUCCESS: Dataset generated at {train_jsonl} ({size_kb:.1f} KB)")
else:
    print(f"\nWARNING: Expected output not found at {train_jsonl}")
    print("Check the command output above for errors.")


In [ ]:
# @title Stage 2: Dataset Sanitization
# @markdown Sanitize the generated dataset: clean ChatML format,
# @markdown remove AI artifacts, validate structure, enforce strict canonical form.

import os
import sys
import subprocess
from pathlib import Path

python_bin = sys.executable
dataset_dir = f"subjects/datasets/{NPC_KEY}/{TECHNIQUE}"
train_jsonl = f"{dataset_dir}/train.jsonl"
clean_jsonl = f"{dataset_dir}/train_clean.jsonl"

print("=" * 60)
print("STAGE 2: Dataset Sanitization")
print("=" * 60)

if not os.path.exists(train_jsonl):
    print(f"ERROR: Raw dataset not found at: {train_jsonl}")
    print("Run Stage 1 (Dataset Generation) first, or place a")
    print("pre-generated dataset at that path.")
    raise SystemExit(1)

print(f"Input:  {train_jsonl}")
print(f"Output: {clean_jsonl}")

cmd = [
    python_bin, "scripts/sanitize_dataset.py",
    train_jsonl,
    "--output", clean_jsonl,
    "--strict-canonical",
]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)

print(result.stdout)

if result.returncode != 0:
    print("=== SANITIZATION FAILED ===")
    print(result.stderr)
    raise SystemExit(1)

if os.path.exists(clean_jsonl):
    size_kb = os.path.getsize(clean_jsonl) / 1024
    print(f"\nSUCCESS: Sanitized dataset at {clean_jsonl} ({size_kb:.1f} KB)")
else:
    print(f"\nWARNING: Clean output not found at {clean_jsonl}")


In [ ]:
# @title Stage 3: DeepEval Quality Gate (Optional)
# @markdown This stage is **optional**. It evaluates dataset quality using
# @markdown DeepEval with a local Ollama judge model.
# @markdown
# @markdown Set `SKIP_QUALITY_GATE = True` below to skip this stage.

import os
import sys
import subprocess
import time
from pathlib import Path

# ---------------------------------------------------------------------------
# Set to True to skip the quality gate entirely:
SKIP_QUALITY_GATE = False
# ---------------------------------------------------------------------------

if SKIP_QUALITY_GATE:
    print("SKIP_QUALITY_GATE is True. Skipping quality evaluation.")
    raise SystemExit(0)

python_bin = sys.executable
dataset_dir = f"subjects/datasets/{NPC_KEY}/{TECHNIQUE}"
clean_jsonl = f"{dataset_dir}/train_clean.jsonl"
spec_path = f"subjects/{NPC_KEY}.json"

print("=" * 60)
print("STAGE 3: DeepEval Quality Gate (Optional)")
print("=" * 60)

if not os.path.exists(clean_jsonl):
    print(f"Clean dataset not found at: {clean_jsonl}")
    print("Run Stage 2 (Sanitization) first.")
    raise SystemExit(0)

print(f"Evaluating: {clean_jsonl}")
print(f"Judge model: {OLLAMA_JUDGE_MODEL}")

# Speed up eval
os.environ["OLLAMA_NUM_PARALLEL"] = "4"

# Ensure Ollama is available
try:
    subprocess.run(["ollama", "--version"], check=True,
                   capture_output=True)
except (FileNotFoundError, subprocess.CalledProcessError):
    print("Ollama not found. Installing...")
    subprocess.run(
        "curl -fsSL https://ollama.com/install.sh | sh",
        shell=True, check=True, timeout=120,
    )

# Ensure Ollama server is running
try:
    subprocess.run(["ollama", "list"], check=True,
                   capture_output=True, timeout=10)
except Exception:
    print("Starting Ollama server...")
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    time.sleep(5)

# Pull judge model
print(f"Pulling judge model: {OLLAMA_JUDGE_MODEL}...")
subprocess.run(["ollama", "pull", OLLAMA_JUDGE_MODEL],
               check=True, timeout=300)

# Run DeepEval
cmd = [
    python_bin, "scripts/dataset_eval.py",
    spec_path,
    "--technique", TECHNIQUE,
    "--judge-model", OLLAMA_JUDGE_MODEL,
    "--cases-per-category", "1",
    "--soft-fail",
]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)

print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[:2000])

quality_summary = f"{dataset_dir}/quality_summary.json"
quality_failures = f"{dataset_dir}/quality_failures.json"
if os.path.exists(quality_summary):
    print(f"\nQuality summary: {quality_summary}")
if os.path.exists(quality_failures):
    print(f"Quality failures: {quality_failures}")

print("\nQuality gate complete.")


In [ ]:
# @title Stage 4: Training + GGUF Export
# @markdown Run Unsloth LoRA fine-tuning and export the trained adapter as a GGUF
# @markdown file for use with Unity / LLMUnity.

import os
import sys
import subprocess
import json
import torch
from pathlib import Path

python_bin = sys.executable
spec_path = f"subjects/{NPC_KEY}.json"
dataset_dir = f"subjects/datasets/{NPC_KEY}/{TECHNIQUE}"
clean_jsonl = f"{dataset_dir}/train_clean.jsonl"

print("=" * 60)
print("STAGE 4: Training + GGUF Export")
print("=" * 60)

# --- VRAM Detection ---
vram_gb = 0.0
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {vram_gb:.2f} GB")
else:
    print("WARNING: No GPU detected. Training will fail without CUDA.")

# --- Check dataset ---
if not os.path.exists(clean_jsonl):
    print(f"Clean dataset not found at: {clean_jsonl}")
    raw_path = f"{dataset_dir}/train.jsonl"
    if os.path.exists(raw_path):
        print("Falling back to raw (unsanitized) dataset.")
        clean_jsonl = raw_path
    else:
        print("ERROR: No dataset found. Run Stages 1 and 2 first.")
        raise SystemExit(1)

print(f"Using dataset: {clean_jsonl}")

# --- VRAM Warning ---
if vram_gb > 0 and vram_gb < 8:
    print(f"\nWARNING: Low VRAM ({vram_gb:.1f} GB).")
    print(f"  Current preset: {TRAINING_PRESET}")
    if TRAINING_PRESET not in ("safe-any", "smoke"):
        print("  Consider using --preset safe-any to avoid OOM.")
        print("  Update TRAINING_PRESET in the config cell and re-run.")

# --- W&B Setup ---
if not os.environ.get("WANDB_API_KEY"):
    try:
        from google.colab import userdata  # type: ignore
        wandb_key = userdata.get("WANDB_API_KEY")
        if wandb_key:
            os.environ["WANDB_API_KEY"] = wandb_key
            print("Loaded WANDB_API_KEY from Colab secrets.")
    except Exception:
        pass

if not os.environ.get("WANDB_API_KEY"):
    os.environ.setdefault("WANDB_MODE", "offline")
    print("W&B API key not found. Using WANDB_MODE=offline.")
    print("Sync later with: wandb sync wandb/offline-*")

# --- Unload Ollama models to free VRAM ---
if TECHNIQUE == "ollama":
    try:
        import urllib.request
        payload = json.dumps({"model": OLLAMA_JUDGE_MODEL,
                              "keep_alive": 0}).encode("utf-8")
        req = urllib.request.Request(
            "http://localhost:11434/api/generate",
            data=payload,
            headers={"Content-Type": "application/json"},
        )
        with urllib.request.urlopen(req, timeout=3):
            print("Unloaded Ollama models from GPU memory.")
    except Exception:
        pass

# --- Train + Export ---
cmd = [
    python_bin, "scripts/train.py",
    spec_path,
    "--from-spec",
    "--technique", TECHNIQUE,
    "--preset", TRAINING_PRESET,
    "--export-gguf",
]
print("\nRunning:", " ".join(cmd))
print("\n" + "=" * 60)
print("TRAINING STARTED (this may take 30-60 minutes)")
print("=" * 60)

result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode != 0:
    print("=== TRAINING FAILED ===")
    # Print last 100 lines of stdout
    stdout_lines = result.stdout.strip().split("\n")
    print("\nLast 100 lines of stdout:")
    print("\n".join(stdout_lines[-100:]))
    print("\nSTDERR (last 50 lines):")
    stderr_lines = result.stderr.strip().split("\n")
    print("\n".join(stderr_lines[-50:]))
    raise SystemExit(1)
else:
    print(result.stdout)
    print("\n" + "=" * 60)
    print("TRAINING + GGUF EXPORT COMPLETE")
    print("=" * 60)

# --- Verify export ---
exports_dir = Path(f"exports/{NPC_KEY}")
if exports_dir.exists():
    gguf_files = sorted(exports_dir.glob("*.gguf"))
    if gguf_files:
        print(f"\nExported GGUF files in {exports_dir}:")
        for f in gguf_files:
            size_mb = f.stat().st_size / 1024**2
            print(f"  {f.name} ({size_mb:.1f} MB)")
    else:
        print(f"\nNo .gguf files found in {exports_dir}.")
else:
    print(f"\nExport directory not found: {exports_dir}")

print("\nRun Stage 5 (Download) to retrieve the GGUF files.")


In [ ]:
# @title Stage 5: Download Artifacts
# @markdown Download exported GGUF models and quality reports.

import os
import glob
from pathlib import Path

print("=" * 60)
print("STAGE 5: Download Artifacts")
print("=" * 60)

# Check runtime
is_colab = False
try:
    from google.colab import files  # type: ignore
    is_colab = True
except ImportError:
    pass

if not is_colab:
    print("Not running in Colab -- download via google.colab.files unavailable.")
    print("Artifacts can be found at:")
    print(f"  exports/{NPC_KEY}/")
    print(f"  subjects/datasets/{NPC_KEY}/{TECHNIQUE}/")
    raise SystemExit(0)

# --- Download GGUF files ---
gguf_dir = f"exports/{NPC_KEY}"
gguf_files = sorted(glob.glob(os.path.join(gguf_dir, "*.gguf")))

if gguf_files:
    print(f"Found {len(gguf_files)} GGUF file(s) in {gguf_dir}:")
    for f in gguf_files:
        size_mb = os.path.getsize(f) / 1024**2
        print(f"  {os.path.basename(f)} ({size_mb:.1f} MB)")

    print("\nDownloading GGUF files...")
    for f in gguf_files:
        print(f"  -> {os.path.basename(f)}")
        files.download(f)
else:
    print(f"No GGUF files found in {gguf_dir}.")
    print("Run Stage 4 (Training + Export) first.")

# --- Download quality reports ---
dataset_dir = f"subjects/datasets/{NPC_KEY}/{TECHNIQUE}"
for report_name in ("quality_summary.json", "quality_failures.json"):
    report_path = os.path.join(dataset_dir, report_name)
    if os.path.exists(report_path):
        print(f"\nDownloading {report_name}...")
        files.download(report_path)

print("\nAll downloads complete.")
